In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from datetime import datetime
from functools import reduce
 
CONFIG = {
    "catalog": "workspace",
    "schema":  "banking_datasentry",
}
 
SCORED_AT = datetime.now().isoformat()
 
# Health status thresholds
THRESHOLDS = {
    "GREEN": 99.0,   # >= 99% pass rate
    "AMBER": 95.0,   # >= 95% pass rate
    "RED":   0.0,    # < 95% pass rate
}
 
spark.sql(f"USE CATALOG {CONFIG['catalog']}")
spark.sql(f"USE SCHEMA {CONFIG['schema']}")
 
print(f"Catalog    : {CONFIG['catalog']}")
print(f"Schema     : {CONFIG['schema']}")
print(f"Scored at  : {SCORED_AT}")
print(f"Thresholds : GREEN >= {THRESHOLDS['GREEN']}% | AMBER >= {THRESHOLDS['AMBER']}% | RED < {THRESHOLDS['AMBER']}%")

In [0]:
#Assigning Health Status
def assign_health_status(pass_rate):
    if pass_rate is None:
        return "RED"
    if pass_rate >= THRESHOLDS["GREEN"]:
        return "GREEN"
    elif pass_rate >= THRESHOLDS["AMBER"]:
        return "AMBER"
    else:
        return "RED"
 
health_status_udf = F.udf(assign_health_status, T.StringType())
 
print("health_status_udf registered.")

In [0]:
#Extracting Rule Scores from a Silver Table
def extract_rule_scores(silver_df, table_name: str, rule_dimension_map: dict):
    """
    Compute pass/fail counts and pass rate for each rule column in silver_df.
    Returns a DataFrame with one row per rule.
    """
    total = silver_df.count()
    rule_cols = [c for c in silver_df.columns if c.startswith("rule_")]
    rows = []
 
    for col in rule_cols:
        rule_name = col.replace("rule_", "")
        pass_count = silver_df.filter(F.col(col) == 1).count()
        fail_count = total - pass_count
        pass_rate  = round((pass_count / total) * 100, 2) if total > 0 else 0.0
        dimension  = rule_dimension_map.get(rule_name, "Unknown")
 
        rows.append((
            table_name,
            rule_name,
            dimension,
            total,
            pass_count,
            fail_count,
            pass_rate,
            SCORED_AT,
        ))
 
    schema = T.StructType([
        T.StructField("table_name",     T.StringType(),  False),
        T.StructField("rule_name",      T.StringType(),  False),
        T.StructField("dimension",      T.StringType(),  False),
        T.StructField("total_records",  T.LongType(),    False),
        T.StructField("pass_count",     T.LongType(),    False),
        T.StructField("fail_count",     T.LongType(),    False),
        T.StructField("pass_rate",      T.DoubleType(),  False),
        T.StructField("scored_at",      T.StringType(),  False),
    ])
 
    return spark.createDataFrame(rows, schema)

In [0]:
#Rule-to-Dimension Maps
customer_rule_dimension_map = {
    "customer_id_not_null":       "Completeness",
    "full_name_not_null":         "Completeness",
    "email_not_null":             "Completeness",
    "created_at_not_null":        "Completeness",
    "email_format_valid":         "Validity",
    "date_of_birth_valid":        "Validity",
    "country_code_valid":         "Validity",
    "customer_id_uuid_format":    "Validity",
    "created_at_not_future":      "Consistency",
    "date_of_birth_not_future":   "Consistency",
    "age_between_18_and_120":     "Consistency",
    "customer_id_unique":         "Uniqueness",
}
 
account_rule_dimension_map = {
    "account_id_not_null":                  "Completeness",
    "customer_id_not_null":                 "Completeness",
    "account_type_not_null":                "Completeness",
    "status_not_null":                      "Completeness",
    "account_type_valid":                   "Validity",
    "status_valid":                         "Validity",
    "currency_valid":                       "Validity",
    "account_id_uuid_format":               "Validity",
    "opened_date_valid":                    "Validity",
    "savings_loan_balance_not_negative":    "Consistency",
    "opened_date_not_future":               "Consistency",
    "account_id_unique":                    "Uniqueness",
    "customer_id_exists_in_customers":      "Referential Integrity",
}
 
transaction_rule_dimension_map = {
    "transaction_id_not_null":              "Completeness",
    "account_id_not_null":                  "Completeness",
    "transaction_type_not_null":            "Completeness",
    "amount_not_null":                      "Completeness",
    "currency_not_null":                    "Completeness",
    "transaction_date_not_null":            "Completeness",
    "status_not_null":                      "Completeness",
    "transaction_type_valid":               "Validity",
    "status_valid":                         "Validity",
    "currency_valid":                       "Validity",
    "transaction_id_uuid_format":           "Validity",
    "amount_greater_than_zero":             "Validity",
    "transaction_date_not_future":          "Consistency",
    "failed_reversed_amount_not_positive":  "Consistency",
    "transaction_id_unique":                "Uniqueness",
    "account_id_exists_in_accounts":        "Referential Integrity",
}
 
print("Rule-to-dimension maps defined.")
print(f"  Customers    : {len(customer_rule_dimension_map)} rules")
print(f"  Accounts     : {len(account_rule_dimension_map)} rules")
print(f"  Transactions : {len(transaction_rule_dimension_map)} rules")

In [0]:
#Loading Silver Tables
silver_customers     = spark.read.format("delta").table("silver_customers")
silver_accounts      = spark.read.format("delta").table("silver_accounts")
silver_transactions  = spark.read.format("delta").table("silver_transactions")
 
print("Silver tables loaded:")
print(f"  silver_customers     : {silver_customers.count():,} rows")
print(f"  silver_accounts      : {silver_accounts.count():,} rows")
print(f"  silver_transactions  : {silver_transactions.count():,} rows")

In [0]:
#Computing Rule Level Scores
customer_rule_scores    = extract_rule_scores(silver_customers,    "customers",    customer_rule_dimension_map)
account_rule_scores     = extract_rule_scores(silver_accounts,     "accounts",     account_rule_dimension_map)
transaction_rule_scores = extract_rule_scores(silver_transactions,  "transactions", transaction_rule_dimension_map)
 
# Union all three into one scorecard
gold_rule_scores = customer_rule_scores \
    .union(account_rule_scores) \
    .union(transaction_rule_scores)
 
# Add health status column
gold_rule_scores = gold_rule_scores.withColumn(
    "health_status",
    health_status_udf(F.col("pass_rate"))
)
 
# Write to Gold Delta table
gold_rule_scores.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_rule_scores")
 
print(f"gold_rule_scores written: {gold_rule_scores.count()} rows")
gold_rule_scores.orderBy("table_name", "dimension", "pass_rate").show(50, truncate=50)

In [0]:
#Computing Dimension Level Scores
gold_dimension_scores = gold_rule_scores \
    .groupBy("table_name", "dimension") \
    .agg(
        F.sum("total_records").alias("total_checks"),
        F.sum("pass_count").alias("total_passes"),
        F.sum("fail_count").alias("total_fails"),
        F.count("rule_name").alias("rule_count"),
    ) \
    .withColumn(
        "pass_rate",
        F.round(F.col("total_passes") / F.col("total_checks") * 100, 2)
    ) \
    .withColumn("scored_at", F.lit(SCORED_AT))
 
# Add health status
gold_dimension_scores = gold_dimension_scores.withColumn(
    "health_status",
    health_status_udf(F.col("pass_rate"))
)
 
# Write to Gold Delta table
gold_dimension_scores.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_dimension_scores")
 
print(f"gold_dimension_scores written: {gold_dimension_scores.count()} rows")
gold_dimension_scores.orderBy("table_name", "pass_rate").show(30, truncate=50)

In [0]:
#Computing Table Level Scores
gold_table_scores = gold_rule_scores \
    .groupBy("table_name") \
    .agg(
        F.sum("total_records").alias("total_checks"),
        F.sum("pass_count").alias("total_passes"),
        F.sum("fail_count").alias("total_fails"),
        F.count("rule_name").alias("total_rules"),
    ) \
    .withColumn(
        "pass_rate",
        F.round(F.col("total_passes") / F.col("total_checks") * 100, 2)
    ) \
    .withColumn("scored_at", F.lit(SCORED_AT))
 
# Add health status per table
gold_table_scores = gold_table_scores.withColumn(
    "health_status",
    health_status_udf(F.col("pass_rate"))
)
 
# Write to Gold Delta table
gold_table_scores.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_table_scores")
 
print(f"gold_table_scores written: {gold_table_scores.count()} rows")
gold_table_scores.orderBy("table_name").show(truncate=50)

In [0]:
#Pipeline Health Summary
print("=" * 60)
print("DATASENTRY — PIPELINE HEALTH SUMMARY")
print("=" * 60)
 
table_scores = gold_table_scores.collect()
total_checks  = sum(r["total_checks"]  for r in table_scores)
total_passes  = sum(r["total_passes"]  for r in table_scores)
total_fails   = sum(r["total_fails"]   for r in table_scores)
overall_rate  = round(total_passes / total_checks * 100, 2)
 
print(f"\nOverall Pipeline Pass Rate : {overall_rate}%")
print(f"Total Checks               : {total_checks:,}")
print(f"Total Passes               : {total_passes:,}")
print(f"Total Fails                : {total_fails:,}")
 
print(f"\n{'TABLE':<20} {'RULES':>6} {'PASS RATE':>10} {'STATUS':>8}")
print("-" * 48)
for r in sorted(table_scores, key=lambda x: x["table_name"]):
    print(f"{r['table_name']:<20} {r['total_rules']:>6} {r['pass_rate']:>9}% {r['health_status']:>8}")
 
print("\nNotebook 04 complete. Gold scoring tables ready for Power BI.")